# Launch records from a saved Wikipedia revision

The parser reads the hash-verified Wikipedia revision 1027686922 saved by the repository input downloader. It extracts numbered Falcon 9 launch rows and checks row counts and identifiers without requesting the live page again.

Run the cells in order from the repository root or `notebooks/`. Install `requirements.txt` and run `download_inputs.py` if `data/` is missing. Local output files are written under `results/`.

## Sources and credit

Adapted from this repository's `scripts/05_web_scraping.py` and the public IBM DS0321EN Skills Network SpaceX labs. IBM and the credited lab authors, including Joseph Santarcangelo, Nayef Abou Tayoun and Yan Luo, retain their source credit. Input URLs and SHA-256 hashes are listed in `SOURCES.json`; see `ATTRIBUTION.md` for third-party terms. This notebook does not grant a new licence over third-party material.


In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 70)

In [2]:
from pathlib import Path

import os

import matplotlib

matplotlib.use('module://matplotlib_inline.backend_inline')

from IPython.display import display

HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / 'SOURCES.json').is_file()), None)

if HERE is None:
    raise FileNotFoundError('Open this notebook from the repository root or its notebooks folder; run download_inputs.py first.')

DATA = HERE / 'data'

OUT = HERE / 'results' / '05_web_scraping'

OUT.mkdir(parents=True, exist_ok=True)

os.chdir(OUT)

import sys

import requests

from bs4 import BeautifulSoup

import re

import unicodedata

import pandas as pd

def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

In [3]:
def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize('NFKD', table_cells.text).strip()
    if mass:
        mass.find('kg')
        new_mass = mass[0:mass.find('kg') + 2]
    else:
        new_mass = 0
    return new_mass

In [4]:
def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not colunm_name.strip().isdigit():
        colunm_name = colunm_name.strip()
        return colunm_name

static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

import json, hashlib

acquisitions = json.loads((HERE / 'DATA_ACQUISITION.json').read_text())

acquisition = next((r for r in acquisitions if r['url'] == static_url))

In [5]:
assert acquisition['status'] == 200 or acquisition.get('source_mode') == 'verified_cache'

html_bytes = (DATA / 'wiki_oldid_1027686922.html').read_bytes()

assert hashlib.sha256(html_bytes).hexdigest() == acquisition['sha256']

print({k: acquisition[k] for k in ['url', 'status', 'source_mode', 'bytes', 'sha256']})

soup = BeautifulSoup(html_bytes, 'html.parser')

soup.title.get_text()

html_tables = soup.find_all('table')

print('Number of HTML tables:', len(html_tables))

first_launch_table = html_tables[2]

print(first_launch_table.get_text(' ', strip=True)[:240])

column_names = []

for header in first_launch_table.find_all('th'):
    name = extract_column_from_header(header)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

launch_dict = dict.fromkeys(column_names)

del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []

launch_dict['Launch site'] = []

{'url': 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922', 'status': None, 'source_mode': 'verified_cache', 'bytes': 3293319, 'sha256': '08fd209f89a5fd2122638a9eea2116d27118437115aee6f310d57b27822e0606'}


Number of HTML tables: 29
Flight No. Date and time ( UTC ) Version, Booster [ b ] Launch site Payload [ c ] Payload mass Orbit Customer Launch outcome Booster landing 1 4 June 2010, 18:45 F9 v1.0 [ 7 ] B0003.1 [ 8 ] CCAFS , SLC-40 Dragon Spacecraft Qualification Uni
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


In [6]:
launch_dict['Payload'] = []

launch_dict['Payload mass'] = []

launch_dict['Orbit'] = []

launch_dict['Customer'] = []

launch_dict['Launch outcome'] = []

launch_dict['Version Booster'] = []

launch_dict['Booster landing'] = []

launch_dict['Date'] = []

launch_dict['Time'] = []

extracted_row = 0

for table_number, table in enumerate(soup.find_all('table', 'wikitable plainrowheaders collapsible')):
    for rows in table.find_all('tr'):
        flight_heading = rows.find('th', scope='row')
        if flight_heading is None:
            continue
        flight_number = flight_heading.get_text(strip=True)
        if not flight_number.isdigit():
            continue
        row = rows.find_all('td', recursive=False)
        if len(row) != 9:
            raise ValueError(f'Unexpected launch row width {flight_number}: {len(row)}')
        extracted_row += 1
        launch_dict['Flight No.'].append(flight_number)
        datatimelist = date_time(row[0])
        launch_dict['Date'].append(datatimelist[0].strip(','))
        launch_dict['Time'].append(datatimelist[1] if len(datatimelist) > 1 else None)
        bv = booster_version(row[1])
        launch_dict['Version Booster'].append(bv or row[1].get_text(' ', strip=True))
        launch_dict['Launch site'].append(row[2].get_text(' ', strip=True))
        launch_dict['Payload'].append(row[3].get_text(' ', strip=True))
        launch_dict['Payload mass'].append(get_mass(row[4]))
        launch_dict['Orbit'].append(row[5].get_text(' ', strip=True))
        launch_dict['Customer'].append(row[6].get_text(' ', strip=True))
        launch_dict['Launch outcome'].append(row[7].get_text(' ', strip=True))
        launch_dict['Booster landing'].append(landing_status(row[8]))

In [7]:
print('Extracted numbered Falcon 9 launches:', extracted_row)

df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})

assert extracted_row > 0

assert all((len(v) == extracted_row for v in launch_dict.values()))

assert df['Flight No.'].is_unique

assert len(df) == extracted_row

raw_numbered_rows = [row for table in soup.find_all('table', 'wikitable plainrowheaders collapsible') for row in table.find_all('tr') if row.find('th', scope='row') is not None and row.find('th', scope='row').get_text(strip=True).isdigit()]

assert len(raw_numbered_rows) == len(df)

assert df.iloc[0]['Date'] == '4 June 2010'

assert df.iloc[0]['Time'] == '18:45'

df.to_csv(OUT / 'spacex_web_scraped.csv', index=False)

display(df.head())

display(df.tail())

result = {'rows': len(df), 'first_flight': df.iloc[0]['Flight No.'], 'last_flight': df.iloc[-1]['Flight No.'], 'all_column_lengths_match': True, 'source_revision': '1027686922', 'scope': 'Numeric Falcon 9 flight identifiers from IBM-specified Wikipedia snapshot; no future operational inference.'}

(OUT / 'RESULTS.json').write_text(json.dumps(result, indent=2))

print(json.dumps(result, indent=2))

Extracted numbered Falcon 9 launches: 121


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),0,LEO ( ISS ),NASA ( COTS ) NRO,Success [ 9 ],F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,"CCAFS , SLC-40",Dragon demo flight C2+ [ 18 ] (Dragon C102),525 kg,LEO ( ISS ),NASA ( COTS ),Success [ 20 ],F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,"CCAFS , SLC-40",SpaceX CRS-1 [ 22 ] (Dragon C103),"4,700 kg",LEO ( ISS ),NASA ( CRS ),Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,"CCAFS , SLC-40",SpaceX CRS-2 [ 22 ] (Dragon C104),"4,877 kg",LEO ( ISS ),NASA ( CRS ),Success,F9 v1.07B0007.1,No,1 March 2013,15:10


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
116,117,"CCSFS , SLC-40",Starlink 27 v1.0 (60 satellites),"15,600 kg",LEO,SpaceX,Success,F9 B5B1051.10,Success,9 May 2021,06:42
117,118,"KSC , LC-39A [ 661 ]",Starlink 26 v1.0 (52 Satellites) Capella -6 & Tyvak -0130 [ 662 ],"~14,000 kg",LEO,SpaceX Capella Space and Tyvak,Success,F9 B5B1058.8,Success,15 May 2021,22:56
118,119,"CCSFS , SLC-40 [ 665 ]",Starlink 28 v1.0 (60 Satellites) [ 665 ],"15,600 kg",LEO,SpaceX,Success,F9 B5B1063.2,Success,26 May 2021,18:59
119,120,"KSC , LC-39A",SpaceX CRS-22 ( Dragon C209.1 ),"3,328 kg",LEO ( ISS ),NASA ( CRS ),Success,F9 B5B1067.1,Success,3 June 2021,17:29
120,121,"CCSFS , SLC-40",SXM-8 [ 674 ],"7,000 kg",GTO,Sirius XM,Success,F9 B5,Success,6 June 2021,04:26


{
  "rows": 121,
  "first_flight": "1",
  "last_flight": "121",
  "all_column_lengths_match": true,
  "source_revision": "1027686922",
  "scope": "Numeric Falcon 9 flight identifiers from IBM-specified Wikipedia snapshot; no future operational inference."
}
